# Experimental policies and observation-driven trajectories

All solver code, policy archives and new figures live inside this folder. Controls are recomputed by Bellman optimization of saved **value functions**, including boundary stopping; saved control tables are not interpolated.

The default constant-noise solvers use the **truncated, renormalized Gaussian** experiment that recovered near-maximum inputs. This discards outside-grid probability and is an intentional historical-behavior experiment. Set `observation_mode="full"` to include all Gaussian probability (analytic linear final jump; extrapolated value-function integration elsewhere). The state-noise solver defaults to full integration.

The first two defaults reproduce the three-observation dynamics parameters; the third reproduces the two-observation sensing parameters. They therefore are not a noise-only comparison unless you match their physical settings and schedules explicitly.

Select one model with `MODEL` below. Only that policy is loaded, simulated and plotted. Set `POLICY_PATH` to load another compatible solver archive directly.

In [ ]:
from pathlib import Path
import sys, json
from dataclasses import asdict
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "workflow.py").exists():
    ROOT = ROOT / "experiments"
if not (ROOT / "workflow.py").exists():
    raise RuntimeError("Run this notebook from the project or experiments directory.")
sys.path.insert(0, str(ROOT))
from workflow import SOLVERS, generate
from policy_plotting import Policy, simulate, plot_heatmaps, plot_trajectories

In [ ]:
# EDIT THIS CELL. Changing solver parameters requires REGENERATE=True.
MODEL = "linear_constant"  # "linear_constant", "nonlinear_constant", or "linear_state"
POLICY_PATH = None  # optional explicit .npz path; None generates/loads the selected default
REGENERATE = False
PARAMETERS = {name: module.Parameters() for name, module in SOLVERS.items()}
# Examples:
# from dataclasses import replace
# PARAMETERS["linear_constant"] = replace(PARAMETERS["linear_constant"], observation_mode="full")
# For a matched two-observation noise comparison:
# PARAMETERS["linear_constant"] = replace(PARAMETERS["linear_constant"],
#     c=0.1, d=0.1, sigma0=0.25, t_obs=(0., 0.2), observation_mode="full")

# Raw observation READINGS, not posterior means. First reading initializes mu.
# Later updates: mu += P/(P+sigma(z)**2) * (reading-mu).
OBSERVATIONS = {
    "linear_constant": [1.10, 1.10, 0.86],
    "nonlinear_constant": [1.10, 1.10, 1.10],
    "linear_state": [0.96, 1.06],
}
INITIAL_STATES = {name: 0.0 for name in SOLVERS}
MAX_TIME = 6.0
# None: actual stage posterior variance for constant noise; sigma(z)^2 at
# t=0 for state noise. Its second observation is the P-independent tail.
# These are slices in (z,mu,P), not averages over all previous observations.
HEATMAP_P = None
EXTRAPOLATE_MU = True  # allow user readings to produce mu outside the saved grid
SAVE_FIGURES = True
# Before-next-observation termination regions (Bellman rollouts on a plotting mesh).
SHOW_REWARD_REGIONS = True
REGION_NZ = 61
REGION_NMU = 81  # increase these two counts for smoother hatch boundaries
HEATMAP_PANEL_WIDTH = 10.0
HEATMAP_PANEL_HEIGHT = 7.0


In [ ]:
if MODEL not in SOLVERS:
    raise ValueError(f"Choose MODEL from {list(SOLVERS)}")
if POLICY_PATH is None:
    path = generate(MODEL, PARAMETERS[MODEL], REGENERATE)
else:
    path = Path(POLICY_PATH).expanduser()
    if not path.is_absolute():
        path = ROOT / path
policy = Policy(path, extrapolate_mu=EXTRAPOLATE_MU)
if policy.model != MODEL:
    raise ValueError(f"Selected MODEL={MODEL}, but archive contains {policy.model}")
policies = {MODEL: policy}
labels = {MODEL: {"linear_constant": "Linear / constant noise",
                  "nonlinear_constant": "Nonlinear / constant noise",
                  "linear_state": "Linear / state noise"}[MODEL]}
saved_parameters = json.loads(str(policy.data["parameters_json"])) if "parameters_json" in policy.data else {
    key: policy.data[key].tolist() for key in ("model", "c", "d", "sigma_obs", "dt", "t_obs")}
display(pd.DataFrame({MODEL: saved_parameters}))
print(MODEL, "->", path.resolve())

## Example trajectories
The system receives each prescribed reading only if it survives to that observation time. The first variance is `sigma(z0)^2`; later updates use the noise at the observation state. Duplicate times in the belief trace show instantaneous updates. A terminal reward stops the trajectory; reaching `MAX_TIME` is reported separately.

In [ ]:
trajectories = {}
for name, policy in policies.items():
    if len(OBSERVATIONS[name]) != len(policy.obs):
        raise ValueError(f"{name}: supply exactly {len(policy.obs)} readings.")
    trajectories[name] = simulate(policy, z0=INITIAL_STATES[name],
                                 observations=OBSERVATIONS[name], max_time=MAX_TIME)
fig_trajectories, _ = plot_trajectories(policies, trajectories, labels)
plt.show()
for name, trajectory in trajectories.items():
    print(name, trajectory["status"])
    print("Observation events:", trajectory["events"])

## Controls immediately after each observation
Each heatmap recomputes the Bellman action at the stated time. Black boundary squares show immediate stopping. Set `SHOW_REWARD_REGIONS=True` for hatched lower/upper reward regions before the next observation. These use Bellman rollouts; exact arrival at the next observation is excluded. No region is drawn after the final observation. For this two-observation state-noise model, the second panel is the stationary tail, whose value is independent of posterior variance.

In [ ]:
fig_heatmaps, _ = plot_heatmaps(policies, labels, P=HEATMAP_P,
    show_reward_regions=SHOW_REWARD_REGIONS, region_nz=REGION_NZ, region_nmu=REGION_NMU,
    panel_width=HEATMAP_PANEL_WIDTH, panel_height=HEATMAP_PANEL_HEIGHT)
plt.show()
if SAVE_FIGURES:
    out = ROOT / "plots" / MODEL
    out.mkdir(parents=True, exist_ok=True)
    for label, fig in [("control_heatmaps", fig_heatmaps), ("example_trajectories", fig_trajectories)]:
        for ext in ("png", "pdf"):
            fig.savefig(out / f"{label}.{ext}", dpi=200, bbox_inches="tight")
    print("Figures:", out)
    for name, trajectory in trajectories.items():
        pd.DataFrame({key: trajectory[key] for key in ("t", "z", "mu", "P")}).to_csv(out / f"{name}_trajectory.csv", index=False)
        pd.DataFrame(trajectory["segments"], columns=["start", "end", "u"]).to_csv(out / f"{name}_controls.csv", index=False)
    configuration = {"parameters": {MODEL: saved_parameters},
                     "readings": {MODEL: OBSERVATIONS[MODEL]}, "initial_states": {MODEL: INITIAL_STATES[MODEL]},
                     "max_time": MAX_TIME,
                     "events": {name: traj["events"] for name, traj in trajectories.items()},
                     "status": {name: traj["status"] for name, traj in trajectories.items()}}
    (out / "configuration.json").write_text(json.dumps(configuration, indent=2))
